# Lab type: review
# Course: ML401 — MLOps & Model Deployment
# Lesson: Kubernetes for ML
# Task: Review the Kubernetes manifests below and answer the open-ended questions about deployment strategy, resource configuration, and probe design for a production ML serving workload.

## The workload

You are deploying a churn prediction model with the following characteristics:

- Model artefact size: **800MB** (loads into ~1.2GB RAM when deserialised)
- Startup time: **~50 seconds** (model deserialisation)
- Prediction latency: **~30ms** per request under normal load
- Expected traffic: **200 req/s** peak, **40 req/s** off-peak
- Each prediction request uses ~100ms of CPU
- SLA: **P99 latency < 200ms**, **availability > 99.9%**

## The manifest to review

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: churn-model
  namespace: ml-serving
spec:
  replicas: 3
  selector:
    matchLabels:
      app: churn-model
  template:
    metadata:
      labels:
        app: churn-model
    spec:
      containers:
      - name: churn-model
        image: registry.example.com/churn-model:v2.1.0
        ports:
        - containerPort: 8080
        resources:
          requests:
            memory: "1Gi"
            cpu: "500m"
          limits:
            memory: "1500Mi"
            cpu: "2000m"
        readinessProbe:
          httpGet:
            path: /health
            port: 8080
          initialDelaySeconds: 10
          periodSeconds: 10
          failureThreshold: 3
        livenessProbe:
          httpGet:
            path: /health
            port: 8080
          initialDelaySeconds: 15
          periodSeconds: 30
          failureThreshold: 3
---
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: churn-model-hpa
  namespace: ml-serving
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: churn-model
  minReplicas: 3
  maxReplicas: 10
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 70
```

## Review question 1: Probe configuration

The model takes ~50 seconds to load. Review the readiness and liveness probe configurations:

```yaml
readinessProbe:
  initialDelaySeconds: 10   # <-- is this correct?
  periodSeconds: 10
  failureThreshold: 3

livenessProbe:
  initialDelaySeconds: 15   # <-- is this correct?
  periodSeconds: 30
  failureThreshold: 3
```

**Answer the following:**

a) What happens to the pod between second 10 and second 50, given the readiness probe configuration above?

b) Is the liveness probe `initialDelaySeconds: 15` safe for a pod that takes 50 seconds to start? What risk does this create?

c) Write corrected probe configurations that account for the 50-second startup time. Justify your chosen values.

**Your answer for Question 1:**

a)

b)

c)
```yaml
# Your corrected probes here:
readinessProbe:

livenessProbe:
```

<details>
<summary>🔑 Reveal answer — Question 1: Probe configuration</summary>

**a) Between second 10 and second 50 (readiness probe):**
The first readiness probe fires at t=10s. The model is still loading (~50s to complete), so `/health` returns non-200. With `failureThreshold: 3` and `periodSeconds: 10`, the pod fails the probe at t=10, t=20, and t=30. After the third failure, Kubernetes removes the pod from the Service endpoints — it stops receiving traffic. The model load continues uninterrupted; once it finishes (~t=50s), the next probe succeeds and the pod is re-added to endpoints. Net result: the pod is unavailable for ~20 seconds during every cold start.

**b) Liveness probe `initialDelaySeconds: 15` — is it safe?**
No. The liveness probe fires at t=15s (model still loading). Probe schedule: fail at t=15, fail at t=45 (still loading), fail at t=75 (by which point the model has loaded and the probe succeeds — so no restart in the happy path). But with any variability (slow storage read, GC pause, cold node), the model may still be loading at t=75s. The third failure triggers `kubectl kill`, the container restarts, and the 50-second load begins again — a restart loop that never resolves. Liveness probes must never fire during a legitimately-in-progress startup.

**c) Corrected probe configurations:**
```yaml
readinessProbe:
  httpGet:
    path: /health
    port: 8080
  initialDelaySeconds: 55   # 50s startup + 5s buffer
  periodSeconds: 10
  failureThreshold: 3

livenessProbe:
  httpGet:
    path: /health
    port: 8080
  initialDelaySeconds: 90   # well past startup; only kills genuinely stuck pods
  periodSeconds: 30
  failureThreshold: 3
```
Readiness `initialDelaySeconds` = startup + buffer so the pod is never removed from endpoints during a normal load. Liveness `initialDelaySeconds` is set much higher — its job is to kill a pod that has *stopped responding after a successful start*, not one that is legitimately loading.

</details>

## Review question 2: Resource limits

The manifest sets:
```yaml
resources:
  requests:
    memory: "1Gi"
    cpu: "500m"
  limits:
    memory: "1500Mi"
    cpu: "2000m"
```

The model loads into ~1.2GB RAM. Peak traffic is 200 req/s; each request uses ~100ms CPU.

**Answer the following:**

a) At peak load, a single pod receives ~67 req/s (200 req/s ÷ 3 pods). Each request uses 100ms CPU. What is the approximate CPU utilisation in millicores per pod at peak load? Is the CPU request (500m) sufficient?

b) The memory request is 1Gi but the model uses 1.2GB. What happens if Kubernetes tries to schedule this pod on a node with only 1Gi of allocatable memory?

c) The memory limit is 1500Mi. If a memory leak causes the pod to use 1600Mi, what happens?

d) Propose revised resource requests and limits based on the workload characteristics. Show your reasoning.

**Your answer for Question 2:**

a)

b)

c)

d)
```yaml
resources:
  requests:
    memory:
    cpu:
  limits:
    memory:
    cpu:
```

<details>
<summary>🔑 Reveal answer — Question 2: Resource limits</summary>

**a) CPU utilisation at peak load:**
67 req/s × 100ms CPU per request = **6700 millicores** of CPU required per pod at peak. The CPU request is 500m — grossly undersized. Even the limit (2000m) is only 30% of what the workload needs. The pod will be heavily CPU-throttled, causing P99 latency to blow past the 200ms SLA. The fix is to increase both the CPU request and let the HPA scale out more pods.

**b) Memory request < actual usage:**
The scheduler uses *requests* for placement. It places the pod on a node with 1Gi allocatable (matching the 1Gi request). Once the model loads (1.2GB ≈ 1229Mi), the pod exceeds its request but is below the 1500Mi limit — no OOMKill. However, the node is now overcommitted: the scheduler's memory accounting is wrong. Other pods on that node may be evicted if the node runs low on real memory. **Fix:** Set memory request to at least 1300Mi (actual footprint + safe headroom).

**c) Memory limit breach (1600Mi > 1500Mi limit):**
The Linux OOM killer fires. Kubernetes marks the pod as `OOMKilled` and restarts it. The pod then goes through the 50-second load time again, creating a brief availability dip. With the corrected liveness `initialDelaySeconds: 90`, the restart won't loop — but the pod is still unavailable for ~50s on each OOMKill event.

**d) Revised resources (with reasoning):**
```yaml
resources:
  requests:
    memory: "1400Mi"   # 1.2GB actual + ~200Mi for request processing overhead
    cpu: "1000m"       # ~off-peak load (13 req/s × 100ms = 1300m; round to 1000m)
  limits:
    memory: "2Gi"      # headroom above request for spikes and GC
    cpu: "4000m"       # peak per pod with 10 replicas: 200 req/s ÷ 10 × 100ms = 2000m;
                        # 4000m leaves headroom; HPA handles scale-out
```
Pair this with raising `maxReplicas` on the HPA: at peak (200 req/s × 100ms = 20 CPU-seconds/s), with a 1000m CPU request and 70% HPA target (700m effective), you need ~29 pods. Set `maxReplicas: 30` or re-examine whether 100ms CPU per request is accurate.

</details>

## Review question 3: Update strategy

The Deployment above has no `strategy` field, so it uses the Kubernetes default (`RollingUpdate` with `maxSurge: 25%`, `maxUnavailable: 25%`).

With 3 replicas, `maxUnavailable: 25%` rounds down to 0 (since 0.75 < 1). So effectively `maxUnavailable: 0` and `maxSurge: 1` (25% of 3, rounded up).

**Answer the following:**

a) A new model version is being deployed. With the default strategy, how many pods will be running simultaneously at the peak of the rolling update? What is the memory footprint on the node at that point?

b) The new model version (v2.2.0) changes the prediction API: it now returns a confidence interval alongside the prediction score. Existing clients do not send the required new request field. Would you use `RollingUpdate` or `Recreate` for this deployment? Justify your choice.

c) After deploying v2.2.0, prediction latency increases from 30ms to 180ms. You want to roll back immediately. Write the `kubectl` command that would roll back this deployment.

**Your answer for Question 3:**

a)

b)

c) `kubectl command:`

<details>
<summary>🔑 Reveal answer — Question 3: Update strategy</summary>

**a) Pods during rolling update:**
Default strategy: `maxSurge: 1` (ceil(25% × 3)), `maxUnavailable: 0` (floor(25% × 3) = 0). Peak: 3 running + 1 new starting = **4 pods simultaneously**. Memory footprint: 4 × ~1.4GB = **~5.6GB RAM** required across hosting nodes at that moment. Ensure node capacity covers the surge — a node that was full at 3 pods will OOM on the 4th.

**b) RollingUpdate vs. Recreate for API-breaking change:**
Use `Recreate` (or API versioning). During a RollingUpdate, v2.1.0 and v2.2.0 pods coexist behind the same Service. Old clients routed to v2.2.0 pods send requests missing the new required field → errors. New clients routed to v2.1.0 pods receive the old response format → breakage. The safest options are: (1) `Recreate` strategy with a short maintenance window, or (2) version the endpoint (`/v2/predict`) so both versions can coexist permanently and clients migrate at their own pace. Never use `RollingUpdate` for backward-incompatible API changes without a versioning strategy.

**c) Immediate rollback command:**
See the command cell below.

</details>

<details>
<summary>🔑 Reveal kubectl rollback commands — Q3c</summary>

```bash
kubectl rollout undo deployment/churn-model -n ml-serving

# Verify rollback status
kubectl rollout status deployment/churn-model -n ml-serving

# Check which version is now running
kubectl get pods -n ml-serving -o jsonpath='{.items[*].spec.containers[0].image}'
```

`kubectl rollout undo` switches traffic back to the previous `ReplicaSet`. The rollback follows the same update strategy as the deployment — a rolling replace back to v2.1.0. Kubernetes retains up to `revisionHistoryLimit` (default: 10) previous ReplicaSets for this purpose.

</details>

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Probe configuration:** Both `initialDelaySeconds` values must exceed the model's startup time; an under-configured liveness probe is especially dangerous — it kills healthy loading containers, creating an OOMKill-like restart loop.

2. **Resource limits:** Memory request must match actual model footprint (not just a round number below it); CPU request must account for concurrent request load using the formula `req/s × CPU-ms-per-request = millicores-required`.

3. **Update strategy:** Default `RollingUpdate` is safe for backward-compatible releases; API-breaking changes require `Recreate` or explicit API versioning to prevent mixed-version traffic errors during rollout. `kubectl rollout undo` provides instant traffic redirection to the previous ReplicaSet.

</details>